# Datenanalyse mit SQL & Python - Tag 3: Übungen

**Teil A:** 30-Minuten Warm-up & Recap zu Tag 2 mit `household_budget`  
**Teil B:** Pandas-Übungen mit einer bereits verbundenen Shop-Tabelle (`shop_joined`)  
**Teil C:** Transferübung mit einer bereits verbundenen Fitness-Tabelle (`fitness_joined`)  
**Ziel:** Erst SQL-Konzepte aus Tag 2 wiederholen, dann mit Pandas Daten prüfen, filtern, gruppieren und visualisieren.


# 09:30–10:30 | Tag-2-Refresh mit 'Household Budget'

## Einrichtung & Daten laden

Führe diese Zelle zuerst aus. Sie lädt:

- `household_budget` als SQL-Tabelle für Teil A
- `shop_joined` als Pandas-DataFrame für Teil B
- `fitness_joined` als Pandas-DataFrame für Teil C


In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

SHOP_DIR = 'https://raw.githubusercontent.com/chiaoya/Data_to_Decision_with_SQL_Python/refs/heads/main/course_data/shop/'
FITNESS_DIR = 'https://raw.githubusercontent.com/chiaoya/Data_to_Decision_with_SQL_Python/refs/heads/main/course_data/fitness/'
BUDGET_PATH = 'https://raw.githubusercontent.com/chiaoya/Data_to_Decision_with_SQL_Python/refs/heads/main/course_data/household_budget.csv'


budget_df = pd.read_csv(BUDGET_PATH)
budget_df['date'] = pd.to_datetime(budget_df['date'])
budget_df['month'] = budget_df['date'].dt.strftime('%Y-%m')

orders = pd.read_csv(SHOP_DIR + 'shop_orders.csv')
customers = pd.read_csv(SHOP_DIR + 'shop_customers.csv')
products = pd.read_csv(SHOP_DIR + 'shop_products.csv')

bookings = pd.read_csv(FITNESS_DIR + 'fitness_bookings.csv')
members = pd.read_csv(FITNESS_DIR + 'fitness_members.csv')
classes = pd.read_csv(FITNESS_DIR + 'fitness_classes.csv')

shop_joined = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(products, on='product_id', how='left')
)
shop_joined['revenue'] = shop_joined['quantity'] * shop_joined['price']

fitness_joined = (
    bookings
    .merge(members, on='member_id', how='left')
    .merge(classes, on='class_id', how='left')
)
fitness_joined['revenue'] = fitness_joined['price']

conn = sqlite3.connect(':memory:')
budget_df.to_sql('budget', conn, index=False, if_exists='replace')

print('SQL-Tabelle budget:', budget_df.shape)
print('Pandas-DataFrame shop_joined:', shop_joined.shape)
print('Pandas-DataFrame fitness_joined:', fitness_joined.shape)


## Teil A: 30-Minuten Warm-up & Recap zu Tag 2 mit Household Budget

In Teil A arbeitest du mit SQL und der Tabelle `budget`.

Diese Tabelle enthält:

- `date`
- `month`
- `category`
- `description`
- `amount`
- `type`

**Ziel:** Wiederhole die wichtigsten Konzepte aus Tag 2:

- `GROUP BY`
- `HAVING`
- Subquery
- CTE mit `WITH`
- Interpretation von aggregierten Ergebnissen


### Übung A1: Tabelle verstehen (ca. 3 Minuten)

**Aufgabe:** Zeige die ersten 10 Zeilen aus `budget` an.


In [ ]:
# Aufgabe: Erste Zeilen anzeigen.
query = '''
SELECT *
FROM budget
LIMIT _____;
'''

result = pd.read_sql_query(query, conn)
result


### Übung A2: GROUP BY nach Kategorie (ca. 5 Minuten)

**Aufgabe:** Berechne die gesamten Ausgaben pro Kategorie.

**Hinweis:** Filtere zuerst auf `type = 'Expense'` und gruppiere nach `category`.


In [ ]:
# Aufgabe: Ausgaben pro Kategorie berechnen.
query = '''
SELECT
    category,
    ABS(SUM(amount)) AS ausgaben
FROM budget
WHERE type = '_____'
GROUP BY _____
ORDER BY ausgaben DESC;
'''

result = pd.read_sql_query(query, conn)
result


### Übung A3: HAVING verwenden (ca. 5 Minuten)

**Aufgabe:** Zeige nur Kategorien mit mehr als 600 Euro Ausgaben.

**Hinweis:** Die Bedingung bezieht sich auf `SUM(amount)`, also brauchst du `HAVING`.


In [ ]:
# Aufgabe: Gruppen mit HAVING filtern.
query = '''
SELECT
    category,
    ABS(SUM(amount)) AS ausgaben
FROM budget
WHERE type = 'Expense'
GROUP BY category
HAVING _____ > 600
ORDER BY ausgaben DESC;
'''

result = pd.read_sql_query(query, conn)
result


### Übung A4: Monatsanalyse mit GROUP BY (ca. 5 Minuten)

**Aufgabe:** Berechne Ausgaben, Einnahmen und Netto-Betrag pro Monat.

**Hinweis:** `amount` ist bei Ausgaben negativ und bei Einnahmen positiv.


In [ ]:
# Aufgabe: Monatswerte berechnen.
query = '''
SELECT
    month,
    ABS(SUM(CASE WHEN type = 'Expense' THEN amount ELSE 0 END)) AS ausgaben,
    SUM(CASE WHEN type = 'Income' THEN amount ELSE 0 END) AS einnahmen,
    SUM(amount) AS netto
FROM budget
GROUP BY _____
ORDER BY month;
'''

result = pd.read_sql_query(query, conn)
result


### Übung A5: Subquery (ca. 6 Minuten)

**Aufgabe:** Welche Kategorien haben höhere Ausgaben als der Durchschnitt aller Kategorien?

**Hinweis:** Die innere Abfrage berechnet zuerst die Ausgaben pro Kategorie.


In [ ]:
# Aufgabe: Kategorien über dem Durchschnitt finden.
query = '''
SELECT
    category,
    ABS(SUM(amount)) AS ausgaben
FROM budget
WHERE type = 'Expense'
GROUP BY category
HAVING ABS(SUM(amount)) > (
    SELECT AVG(kategorie_ausgaben)
    FROM (
        SELECT ABS(SUM(amount)) AS kategorie_ausgaben
        FROM budget
        WHERE type = 'Expense'
        GROUP BY category
    )
)
ORDER BY ausgaben DESC;
'''

result = pd.read_sql_query(query, conn)
result


### Übung A6: CTE als Vergleich (ca. 6 Minuten)

**Aufgabe:** Löse A5 noch einmal mit einer CTE.

**Vergleich:** Welche Version ist leichter zu lesen: Subquery oder CTE?


In [ ]:
# Aufgabe: Gleiche Logik mit WITH schreiben.
query = '''
WITH category_expenses AS (
    SELECT
        category,
        ABS(SUM(amount)) AS ausgaben
    FROM budget
    WHERE type = 'Expense'
    GROUP BY category
),
average_expense AS (
    SELECT AVG(ausgaben) AS durchschnitt
    FROM category_expenses
)
SELECT
    ce.category,
    ce.ausgaben
FROM category_expenses AS ce
CROSS JOIN average_expense AS ae
WHERE ce.ausgaben > ae.durchschnitt
ORDER BY ce.ausgaben DESC;
'''

result = pd.read_sql_query(query, conn)
result


### Reflexion Teil A (ca. 3 Minuten)

Diskutiert kurz:

1. Wann brauchst du `HAVING` statt `WHERE`?
1. Warum kann eine CTE lesbarer sein als eine Subquery?
1. Welche Kategorie oder welcher Monat wäre aus Budget-Sicht kritisch?


# 14:20–15:40 | Praxis mit Day_3_Exercise_Teil B & C

## Teil B: Shop-Daten mit Pandas

Du arbeitest mit `shop_joined`. Diese Tabelle enthält Bestellungen, Kundendaten und Produktdaten in einem DataFrame.

Wichtige Spalten:

- `order_id`
- `customer_id`
- `name`
- `city`
- `product_name`
- `category`
- `quantity`
- `price`
- `revenue`


### Übung B1: DataFrame verstehen

**Aufgabe:** Zeige die ersten Zeilen, die Spaltennamen und die Datentypen von `shop_joined` an.


In [ ]:
# Aufgabe: Erste Orientierung im DataFrame.
shop_joined._____()


In [ ]:
# Aufgabe: Spaltennamen anzeigen.
shop_joined._____


In [ ]:
# Aufgabe: Datentypen und fehlende Werte prüfen.
shop_joined._____()


### Übung B2: Filtern

**Aufgabe:** Filtere alle Bestellungen aus der Stadt `Berlin`.


In [ ]:
# Aufgabe: Filtere Bestellungen aus Berlin.
berlin_orders = shop_joined[shop_joined['_____'] == 'Berlin']
berlin_orders.head()


### Übung B3: Neue Kennzahl prüfen

**Aufgabe:** Zeige `quantity`, `price` und `revenue`. Prüfe, ob `revenue = quantity * price` sinnvoll berechnet wurde.


In [ ]:
# Aufgabe: Relevante Spalten auswählen.
shop_joined[['_____', '_____', '_____']].head(10)


### Übung B4: Umsatz pro Stadt

**Aufgabe:** Berechne den gesamten Umsatz pro Stadt.

**Hinweis:** Nutze `groupby`, wähle `revenue` und summiere.


In [ ]:
# Aufgabe: Umsatz pro Stadt berechnen.
revenue_by_city = (
    shop_joined
    .groupby('_____')['_____']
    .sum()
    .sort_values(ascending=False)
)
revenue_by_city


### Übung B5: Umsatz pro Produktkategorie

**Aufgabe:** Welche Produktkategorie bringt den höchsten Umsatz?


In [ ]:
# Aufgabe: Umsatz pro Kategorie berechnen.
revenue_by_category = (
    shop_joined
    .groupby('_____')['revenue']
    .sum()
    .sort_values(ascending=False)
)
revenue_by_category


### Übung B6: Top 5 Bestellungen

**Aufgabe:** Zeige die fünf Bestellungen mit dem höchsten Umsatz.


In [ ]:
# Aufgabe: Nach revenue sortieren und Top 5 anzeigen.
top_orders = shop_joined.sort_values(by='_____', ascending=False).head(_____)
top_orders[['order_id', 'name', 'city', 'product_name', 'quantity', 'price', 'revenue']]


### Übung B7: Visualisierung

**Aufgabe:** Erstelle ein Balkendiagramm für den Umsatz pro Produktkategorie.


In [ ]:
# Aufgabe: Balkendiagramm mit Seaborn erstellen.
category_plot_data = revenue_by_category.reset_index()
category_plot_data.columns = ['category', 'revenue']

sns.barplot(data=category_plot_data, x='revenue', y='category')
plt.title('Umsatz pro Produktkategorie')
plt.xlabel('Umsatz')
plt.ylabel('Kategorie')
plt.show()


### Reflexion Teil B

Diskutiert kurz:

1. Welche Stadt ist im Shop-Datensatz besonders wichtig?
1. Welche Produktkategorie ist umsatzstark?
1. Welche zusätzliche Information wäre für eine bessere Shop-Analyse hilfreich?


## Teil C: Fitness-Daten als Transferübung

Du arbeitest mit `fitness_joined`. Diese Tabelle enthält Buchungen, Mitgliedsdaten und Kursdaten in einem DataFrame.

Wichtige Spalten:

- `booking_id`
- `member_id`
- `name`
- `level`
- `class_name`
- `trainer`
- `price`
- `revenue`


### Übung C1: DataFrame verstehen

**Aufgabe:** Zeige die ersten Zeilen, die Spaltennamen und die Datentypen von `fitness_joined` an.


In [ ]:
# Aufgabe: Erste Orientierung im Fitness-DataFrame.
fitness_joined._____()


In [ ]:
# Aufgabe: Spaltennamen anzeigen.
fitness_joined._____


In [ ]:
# Aufgabe: Datentypen und fehlende Werte prüfen.
fitness_joined._____()


### Übung C2: Filtern

**Aufgabe:** Filtere alle Buchungen von Mitgliedern mit Level `Advanced`.


In [ ]:
# Aufgabe: Filtere Advanced-Mitglieder.
advanced_bookings = fitness_joined[fitness_joined['_____'] == 'Advanced']
advanced_bookings.head()


### Übung C3: Buchungen pro Level

**Aufgabe:** Wie viele Buchungen gibt es pro Mitgliedslevel?


In [ ]:
# Aufgabe: Buchungen pro Level zählen.
bookings_by_level = (
    fitness_joined
    .groupby('_____')['booking_id']
    .count()
    .sort_values(ascending=False)
)
bookings_by_level


### Übung C4: Umsatz pro Trainerin oder Trainer

**Aufgabe:** Berechne den Umsatz pro Trainerin oder Trainer.


In [ ]:
# Aufgabe: Revenue pro Trainer berechnen.
revenue_by_trainer = (
    fitness_joined
    .groupby('_____')['_____']
    .sum()
    .sort_values(ascending=False)
)
revenue_by_trainer


### Übung C5: Beliebteste Kurse

**Aufgabe:** Welche fünf Kurse wurden am häufigsten gebucht?


In [ ]:
# Aufgabe: Top 5 Kurse nach Buchungsanzahl finden.
top_classes = (
    fitness_joined
    .groupby('_____')['booking_id']
    .count()
    .sort_values(ascending=False)
    .head(_____)
)
top_classes


### Übung C6: Durchschnittlicher Preis pro Level

**Aufgabe:** Berechne den durchschnittlichen Kurspreis pro Mitgliedslevel.


In [ ]:
# Aufgabe: Durchschnittspreis pro Level berechnen.
avg_price_by_level = (
    fitness_joined
    .groupby('_____')['_____']
    .mean()
    .sort_values(ascending=False)
)
avg_price_by_level


### Übung C7: Visualisierung

**Aufgabe:** Erstelle ein Balkendiagramm für den Umsatz pro Trainerin oder Trainer.


In [ ]:
# Aufgabe: Balkendiagramm mit Seaborn erstellen.
trainer_plot_data = revenue_by_trainer.reset_index()
trainer_plot_data.columns = ['trainer', 'revenue']

sns.barplot(data=trainer_plot_data, x='revenue', y='trainer')
plt.title('Umsatz pro Trainerin oder Trainer')
plt.xlabel('Umsatz')
plt.ylabel('Trainerin oder Trainer')
plt.show()


### Reflexion Teil C

Diskutiert kurz:

1. Welches Mitgliedslevel ist besonders aktiv?
1. Welche Trainerinnen oder Trainer bringen viel Umsatz?
1. Welche Analyse war ähnlich wie in Teil B mit den Shop-Daten?


## Abschluss

Heute verbindest du drei Denkweisen:

- Teil A wiederholt SQL-Konzepte aus Tag 2.
- Teil B überträgt Analysefragen auf Pandas mit Shop-Daten.
- Teil C zeigt, dass dieselbe Pandas-Logik auch mit Fitness-Daten funktioniert.

Der wichtigste Schritt ist nicht nur der Code, sondern die Frage: Welche Entscheidung kann ich mit dem Ergebnis besser treffen?


## Musterlösungen

Die folgenden Lösungen zeigen jeweils ein mögliches sauberes Muster. Bei Visualisierungen und Interpretationen sind andere gute Varianten möglich.


### Lösungen Teil A: SQL-Recap

In [ ]:
sql_loesungen = {
    'A1': 'SELECT * FROM budget LIMIT 10;',
    'A2': '''
        SELECT category, SUM(amount) AS ausgaben
        FROM budget
        WHERE type = 'Expense'
        GROUP BY category
        ORDER BY ausgaben DESC;
    ''',
    'A3': '''
        SELECT category, SUM(amount) AS ausgaben
        FROM budget
        WHERE type = 'Expense'
        GROUP BY category
        HAVING SUM(amount) > 100
        ORDER BY ausgaben DESC;
    ''',
    'A4': '''
        SELECT month, SUM(amount) AS betrag
        FROM budget
        GROUP BY month
        ORDER BY month;
    ''',
    'A5': '''
        SELECT category, SUM(amount) AS ausgaben
        FROM budget
        WHERE type = 'Expense'
        GROUP BY category
        HAVING SUM(amount) > (
            SELECT AVG(category_total)
            FROM (
                SELECT SUM(amount) AS category_total
                FROM budget
                WHERE type = 'Expense'
                GROUP BY category
            )
        )
        ORDER BY ausgaben DESC;
    ''',
    'A6': '''
        WITH category_totals AS (
            SELECT category, SUM(amount) AS ausgaben
            FROM budget
            WHERE type = 'Expense'
            GROUP BY category
        ),
        average_total AS (
            SELECT AVG(ausgaben) AS avg_ausgaben
            FROM category_totals
        )
        SELECT category, ausgaben
        FROM category_totals
        CROSS JOIN average_total
        WHERE ausgaben > avg_ausgaben
        ORDER BY ausgaben DESC;
    ''',
}

for name, query in sql_loesungen.items():
    print(f'--- {name} ---')
    display(pd.read_sql_query(query, conn).head(10))


### Lösungen Teil B: Shop-Daten mit Pandas

In [ ]:
# B1
shop_joined.head()
shop_joined.columns
shop_joined.info()

# B2
berlin_orders = shop_joined.loc[shop_joined['city'] == 'Berlin']
berlin_orders.head()

# B3
shop_joined[['order_id', 'customer_id', 'city', 'product_name', 'quantity', 'price', 'revenue']].head()

# B4
revenue_by_city = (
    shop_joined
    .groupby('city')['revenue']
    .sum()
    .sort_values(ascending=False)
)
revenue_by_city

# B5
revenue_by_category = (
    shop_joined
    .groupby('category')['revenue']
    .sum()
    .sort_values(ascending=False)
)
revenue_by_category

# B6
top_orders = shop_joined.sort_values('revenue', ascending=False).head(5)
top_orders


In [ ]:
# B7
plot_data = revenue_by_category.reset_index()
plot_data.columns = ['category', 'revenue']

sns.barplot(data=plot_data, x='revenue', y='category')
plt.title('Umsatz pro Produktkategorie')
plt.xlabel('Umsatz')
plt.ylabel('Kategorie')
plt.show()


### Lösungen Teil C: Fitness-Daten mit Pandas

In [ ]:
# C1
fitness_joined.head()
fitness_joined.columns
fitness_joined.info()

# C2
advanced_members = fitness_joined.loc[fitness_joined['level'] == 'Advanced']
advanced_members.head()

# C3
bookings_by_level = fitness_joined['level'].value_counts()
bookings_by_level

# C4
revenue_by_trainer = (
    fitness_joined
    .groupby('trainer')['revenue']
    .sum()
    .sort_values(ascending=False)
)
revenue_by_trainer

# C5
top_classes = fitness_joined['class_name'].value_counts().head(5)
top_classes

# C6
avg_price_by_level = (
    fitness_joined
    .groupby('level')['price']
    .mean()
    .sort_values(ascending=False)
)
avg_price_by_level


In [ ]:
# C7
trainer_plot_data = revenue_by_trainer.reset_index()
trainer_plot_data.columns = ['trainer', 'revenue']

sns.barplot(data=trainer_plot_data, x='revenue', y='trainer')
plt.title('Umsatz pro Trainerin oder Trainer')
plt.xlabel('Umsatz')
plt.ylabel('Trainerin oder Trainer')
plt.show()
